## Waterfall: Single-Position Token Intervention Sweep

Intervenes at a single token position, does one forward pass, and records the predicted next token plus factual/counterfactual answer probabilities. Meant to be re-run across a range of intervention positions to see how the answer probability shifts as the intervention point moves further into the reasoning chain.

In [1]:
%load_ext autoreload
%autoreload 2

### Set-up

In [2]:
import sys
sys.path.append("src")

import torch
import gc
from tqdm import tqdm

import _config
import _util

In [3]:
_util.print_GPU_availbility()

CUDA is available: True
Available devices:
  GPU 0: NVIDIA RTX A6000
|===========================================================================|
|                  PyTorch CUDA memory summary, device ID 0                 |
|---------------------------------------------------------------------------|
|            CUDA OOMs: 0            |        cudaMalloc retries: 0         |
|===========================================================================|
|        Metric         | Cur Usage  | Peak Usage | Tot Alloc  | Tot Freed  |
|---------------------------------------------------------------------------|
| Allocated memory      |      0 B   |      0 B   |      0 B   |      0 B   |
|       from large pool |      0 B   |      0 B   |      0 B   |      0 B   |
|       from small pool |      0 B   |      0 B   |      0 B   |      0 B   |
|---------------------------------------------------------------------------|
| Active memory         |      0 B   |      0 B   |      0 B   |      0 B

In [4]:
prompt_config = _config.PromptConfig(
    model_type="GPT-OSS_vanilla", # GPT-OSS or R1
    prompt_type="h", # empty or pre_result or pre_sum
)
intervention_config = _config.InterventionConfig(
    intervention_loc="explicit_ids",
    intervention_ids=[11],
)
run_config = _config.RunConfig(
    experiment_root="experiments/token_intervention",
    result_dir="steps",
    output_filename=f"{prompt_config.stem}_{'_'.join(str(id) for id in intervention_config.intervention_ids)}.csv",
)
batch_size = 24

model, tokenizer = _config.load_model(prompt_config.model_type)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [9]:
prompts = _config.load_prompts(prompt_config)
print(f"loaded {len(prompts)} prompts")

loaded 256 divided prompts


In [10]:
intervention_ids = list(intervention_config.intervention_ids)
print(intervention_ids)

[11]


### Run: single-position intervention + 1-token prediction

Builds the intervened prompt, does a single forward pass, and reads off the predicted next token plus factual/counterfactual answer probabilities directly from the logits.

In [11]:
# Get header of prompts dataset
header = list(prompts.columns) + ['intervention_ids', 'intervention_id', 'intervend_prompt', 'generated_text', 'factual_label_probability', 'counterfactual_label_probability']
filepath = _config.build_run_output_filepath(prompt_config, run_config, header)

for i in tqdm(range(0, len(prompts), batch_size)):
    torch.cuda.empty_cache()
    gc.collect()
    batch_rows = prompts.iloc[i:i+batch_size]

    factual_labels, counterfactual_labels = _config.prepare_label_tensors(tokenizer, batch_rows)
    intervention_prompts = _config.build_intervened_prompts(batch_rows, intervention_ids)
    
    # Tokenize all prompts in the batch
    tokens = tokenizer(intervention_prompts, add_special_tokens=False, return_tensors="pt", padding=True, padding_side="left").to(model.device)
    input_length = tokens.input_ids.shape[1]

    # Forward pass for the entire batch
    with torch.no_grad():
        output = model(input_ids=tokens.input_ids, attention_mask=tokens.attention_mask)

    pred_toks = output.logits[:,-1,:].argmax(dim=-1)
    prob = torch.nn.functional.softmax(output.logits[:,-1,:], dim=-1)
    factual_prob = prob[torch.arange(prob.shape[0]), factual_labels]
    counterfactual_prob = prob[torch.arange(prob.shape[0]), counterfactual_labels]
    tokens["input_ids"] = torch.cat([tokens["input_ids"], pred_toks.unsqueeze(-1)], dim=1)
    del output
    
    # Process each generated text in the batch
    for j, (_, row) in enumerate(batch_rows.iterrows()):
        intervention_prompt = intervention_prompts[j]
        generated_text = tokenizer.decode(tokens["input_ids"][j,input_length:]).replace(tokenizer.pad_token[-1], "")
        _config.write_to_csv(filepath, row.to_list() + [intervention_ids, ', '.join(str(id) for id in intervention_ids), intervention_prompt, generated_text.replace(tokenizer.pad_token[-1], ""), factual_prob[j].item(), counterfactual_prob[j].item()])


100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 11/11 [00:27<00:00,  2.49s/it]


### (Superseded) old single-token intervention cell

Left in place for reference only — an earlier version of the run cell that no longer matches this prompt schema and should not be executed.

In [7]:
# Superseded by the refactored run cell above. The old duplicate version expected
# factual_output/counterfactual_output columns that are not present in this prompt file.


  0%|                                                                                                                        | 0/11 [00:00<?, ?it/s]


KeyError: 'factual_output'